# 🤖 Colab LLM Server → Hermes (via Cloudflare Tunnel)

يشغّل موديل **Qwen2.5-Coder-7B-Instruct (AWQ)** على GPU مجاني بواجهة متوافقة مع OpenAI،
ويعرضه على `https://colab-llm.cloudstars.club/v1` عبر Cloudflare Tunnel ثابت.

## قبل التشغيل (مرة واحدة):
1. **Runtime → Change runtime type → T4 GPU** ثم Save.
2. من الشريط الجانبي دوس أيقونة المفتاح 🔑 (**Secrets**) وضيف:
   - `CF_TUNNEL_TOKEN` = توكن تونل `colab-llm` (اللي يبدأ بـ `eyJ...`) — وفعّل **Notebook access**.
   - `LLM_API_KEY` = أي مفتاح سرّي تختاره (لحماية الـ endpoint) — لو سيبته فاضي هيتولّد واحد تلقائي.
3. دوس **Runtime → Run all** — وخلاص.

> اترك التبويب مفتوح؛ قفل التبويب أو خمول ~90 دقيقة يوقف الجلسة.

### 1️⃣ تأكد من الـ GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "⚠️ مفيش GPU! روح Runtime > Change runtime type > T4 GPU"
print("✅ GPU:", torch.cuda.get_device_name(0))

### 2️⃣ تثبيت vLLM + cloudflared (~3-5 دقائق)

In [ ]:
# vLLM (محرك التقديم السريع بواجهة OpenAI)
!pip install -q vllm 2>&1 | tail -3

# cloudflared (للنفق الثابت)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print("\n✅ تم التثبيت. لو ظهر تعارض في torch: Runtime > Restart session ثم Run all تاني.")

### 3️⃣ الإعدادات والمفاتيح

In [ ]:
from google.colab import userdata
import secrets

# ===== الموديل =====
MODEL       = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"  # توازن سرعة/جودة ممتاز على T4
SERVED_NAME = "qwen2.5-coder"                        # الاسم اللي هتستخدمه في Hermes
PORT        = 8000
MAX_LEN     = 8192                                     # طوّله لو الـ VRAM سمح (16384/32768)

# ===== المفاتيح (من Colab Secrets 🔑) =====
CF_TUNNEL_TOKEN = userdata.get('CF_TUNNEL_TOKEN')
try:
    LLM_API_KEY = userdata.get('LLM_API_KEY')
except Exception:
    LLM_API_KEY = None
if not LLM_API_KEY:
    LLM_API_KEY = "sk-colab-" + secrets.token_hex(16)

assert CF_TUNNEL_TOKEN, "⚠️ ضيف CF_TUNNEL_TOKEN في Colab Secrets (🔑 على الشمال)"
print("✅ SERVED_NAME =", SERVED_NAME)
print("🔑 LLM_API_KEY =", LLM_API_KEY, "  (حطّه في إعدادات Hermes)")

### 4️⃣ تشغيل سيرفر الموديل (vLLM) وانتظار جاهزيته

In [ ]:
import subprocess, time, requests

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--served-model-name", SERVED_NAME,
    "--dtype", "half",
    "--quantization", "awq",
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", "0.90",
    "--port", str(PORT),
    "--api-key", LLM_API_KEY,
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
]
vllm_log = open("/content/vllm.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
print("⏳ vLLM بيحمّل الموديل... (pid", vllm_proc.pid, ") — أول مرة بينزّل الموديل فاصبر 2-4 دقائق")

ready = False
for i in range(180):
    try:
        r = requests.get(f"http://localhost:{PORT}/v1/models",
                         headers={"Authorization": f"Bearer {LLM_API_KEY}"}, timeout=3)
        if r.status_code == 200:
            print("\n✅ vLLM جاهز:", r.json().get("data", [{}])[0].get("id"))
            ready = True
            break
    except Exception:
        pass
    if vllm_proc.poll() is not None:
        print("\n❌ vLLM وقف. آخر اللوج:\n", open('/content/vllm.log').read()[-3000:])
        break
    time.sleep(5)
if not ready and vllm_proc.poll() is None:
    print("\n⏰ طال التحميل. آخر اللوج:\n", open('/content/vllm.log').read()[-3000:])

### 5️⃣ تشغيل Cloudflare Tunnel

In [ ]:
import subprocess, time
cf_log = open("/content/cloudflared.log", "w")
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "run", "--token", CF_TUNNEL_TOKEN],
                           stdout=cf_log, stderr=subprocess.STDOUT)
print("⏳ cloudflared بيتصل... (pid", cf_proc.pid, ")")
time.sleep(10)
print(open('/content/cloudflared.log').read()[-1500:])

### 6️⃣ اختبار الـ endpoint العام

In [ ]:
import requests
PUBLIC_URL = "https://colab-llm.cloudstars.club"
try:
    r = requests.post(f"{PUBLIC_URL}/v1/chat/completions",
        headers={"Authorization": f"Bearer {LLM_API_KEY}", "Content-Type": "application/json"},
        json={"model": SERVED_NAME,
              "messages": [{"role": "user", "content": "قول أهلاً في 3 كلمات"}],
              "max_tokens": 30},
        timeout=60)
    print("HTTP", r.status_code)
    print(r.json()["choices"][0]["message"]["content"] if r.status_code == 200 else r.text[:500])
except Exception as e:
    print("❌ خطأ:", e, "\nاتأكد إن الخليتين 4 و 5 اشتغلوا بنجاح.")

### 7️⃣ ابقّ الجلسة شغّالة (اترك هذه الخلية تعمل)

In [ ]:
import time, datetime
print("🟢 الخدمة شغّالة على:", "https://colab-llm.cloudstars.club/v1")
print("   اقفل هذه الخلية لإيقاف الخدمة.\n")
while True:
    up_vllm = 'OK' if vllm_proc.poll() is None else 'DOWN'
    up_cf   = 'OK' if cf_proc.poll() is None else 'DOWN'
    print(f"[{datetime.datetime.now():%H:%M:%S}] vLLM={up_vllm}  tunnel={up_cf}")
    if up_vllm == 'DOWN' or up_cf == 'DOWN':
        print("⚠️ احد العمليتين وقف — راجع اللوج وأعد تشغيل الخلية المعنية.")
        break
    time.sleep(60)

---
## 🔌 ربطه مع Hermes / opencode

استخدم القيم دي كـ provider متوافق مع OpenAI:

| الحقل | القيمة |
|------|-------|
| **Base URL** | `https://colab-llm.cloudstars.club/v1` |
| **API Key** | قيمة `LLM_API_KEY` (مطبوعة في خلية 3) |
| **Model** | `qwen2.5-coder` |

خلّي Hermes/opencode يرجع تلقائيًا لـ DeepSeek Flash لو الـ endpoint ده وقع (fallback).